In [1]:
import numpy as np
import pandas as pd

subjects = ['01', '09', '10', '13', '16', '17']
window_size = 410
step_size = 205

all_extracted_features = []

for sub in subjects:
    print(f"Processing subject {sub}...")
    file_name = f"../Data/{sub}_filtered_all_data.pkl"
    df = pd.read_pickle(file_name)
    
    emg_cols = [col for col in df.columns if 'EMG' in col]
    imu_cols = [col for col in df.columns if '-' in col]
    gonio_cols = [col for col in df.columns if col.startswith('G_')]
    
    for start in range(0, len(df) - window_size + 1, step_size):
        end = start + window_size
        window = df.iloc[start:end]
        
        win_feats = {}
        
        # Metadata
        win_feats['Subject'] = sub
        win_feats['Timestamp'] = window['Timestamp'].iloc[-1]
        win_feats['Activity'] = window['Activity'].iloc[-1]
        win_feats['Gait'] = window['Gait'].iloc[-1]
        win_feats['Reps'] = window['Reps'].iloc[-1]
        
        # EMG features
        emg_data = window[emg_cols].values.astype(float)
        emg_mav = np.mean(np.abs(emg_data), axis=0)
        emg_rms = np.sqrt(np.mean(emg_data**2, axis=0))
        
        for i, col in enumerate(emg_cols):
            win_feats[f"{col}_MAV"] = emg_mav[i]
            win_feats[f"{col}_RMS"] = emg_rms[i]
            
        # IMU features
        imu_data = window[imu_cols].values.astype(float)
        imu_mean = np.mean(imu_data, axis=0)
        imu_std = np.std(imu_data, axis=0)
        
        for i, col in enumerate(imu_cols):
            win_feats[f"IMU_{col}_Mean"] = imu_mean[i]
            win_feats[f"IMU_{col}_Std"] = imu_std[i]
            
        # Goniometer targets
        gonio_data = window[gonio_cols].values.astype(float)
        gonio_mean = np.mean(gonio_data, axis=0)
        
        for i, col in enumerate(gonio_cols):
            win_feats[f"{col}_Target"] = gonio_mean[i]
            
        all_extracted_features.append(win_feats)

feature_df = pd.DataFrame(all_extracted_features)

feature_df.to_pickle("../Data/feature_database.pkl")
feature_df.to_csv("../Data/feature_database.csv", index=False)
print("Saved to ../Data/feature_database.pkl and ../Data/feature_database.csv")

feature_df

Processing subject 01...
Processing subject 09...
Processing subject 10...
Processing subject 13...
Processing subject 16...
Processing subject 17...
Saved to ../Data/feature_database.pkl and ../Data/feature_database.csv


In [2]:
feature_df

,Subject,Timestamp,Activity,Gait,Reps,EMG_TA_CH_0_MAV,EMG_TA_CH_0_RMS,EMG_TA_CH_1_MAV,EMG_TA_CH_1_RMS,EMG_TA_CH_2_MAV,...,IMU_gY-h_Mean,IMU_gY-h_Std,IMU_gZ-h_Mean,IMU_gZ-h_Std,G_DA_X_Target,G_DA_Y_Target,G_DK_X_Target,G_DK_Y_Target,G_NDA_X_Target,G_NDA_Y_Target
0,01,NaN,walk,27.194149,0.0,0.0,0.0,0.017571,0.022603,0.000000,...,-0.007882,0.809663,-0.076544,0.797510,-9.294385,-11.065770,17.837338,-9.626089,-0.219076,-9.245965
1,01,NaN,walk,40.824468,0.0,0.0,0.0,0.010253,0.015271,0.000000,...,-0.156689,0.682850,0.621467,0.460594,-8.962225,-10.373013,17.695806,-7.825833,-0.432420,-7.549948
2,01,NaN,walk,54.454787,0.0,0.0,0.0,0.015024,0.024944,0.000000,...,0.124557,0.294443,0.012496,0.768508,-8.435269,-11.451688,14.909027,-8.787124,0.338288,-7.137809
3,01,NaN,walk,68.085106,0.0,0.0,0.0,0.033903,0.046292,0.000000,...,0.159321,0.495760,-0.057165,0.703800,-7.180642,-13.218277,30.495154,-12.084844,1.598074,-7.810472
4,01,NaN,walk,81.715426,0.0,0.0,0.0,0.039538,0.050389,0.000000,...,-0.335907,0.606948,-0.052964,0.675900,-6.788042,-11.529752,49.108581,-16.516739,2.031915,-7.500499
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37105,17,NaN,sts_,75.722394,9.0,0.0,0.0,0.013587,0.015818,0.011678,...,0.613416,0.069251,-0.006969,0.025613,-7.899140,-0.576262,65.374717,-19.206143,1.399649,-4.337703
37106,17,NaN,sts_,81.011352,9.0,0.0,0.0,0.012219,0.015141,0.013516,...,0.468179,0.231575,-0.015466,0.024551,-7.840832,-0.680289,65.473076,-19.163369,1.566042,-4.693531
37107,17,NaN,sts_,86.300310,9.0,0.0,0.0,0.008105,0.011170,0.010617,...,0.193768,0.163525,-0.017703,0.016463,-7.750874,-0.782994,65.443459,-19.461595,1.673369,-4.798943
37108,17,NaN,sts_,91.589267,9.0,0.0,0.0,0.004600,0.005801,0.007988,...,0.062749,0.071171,-0.012843,0.014260,-7.729427,-0.814135,65.489721,-19.649431,1.626477,-4.745062
